# Reproducing Kim & Shin (2025) on its main dataset: Allen Visual Coding Neuropixels natural scenes

Paper: Kim, J. H. & Shin, H. (2025), bioRxiv doi:10.1101/2025.06.30.662469 (July 2025 preprint, the text the claims file was written from).

The registry tests this paper on dandiset 000248, which the paper uses only for pictures (first notebook). Its findings rest on the Allen Brain Observatory Visual Coding Neuropixels data, "Brain Observatory 1.1" sessions, which the paper reads through AllenSDK without naming a release. These claims are `data: elsewhere`. Here they are tested on the same sessions as deposited on DANDI: dandiset 000021, "Allen Institute - Visual Coding - Neuropixels (Brain Observatory 1.1 Stimulus Set)", version 0.251116.2246 (doi:10.48324/dandi.000021/0.251116.2246), the only published version, streamed from the archive. The 32 session files it holds match the paper's 32 Brain Observatory 1.1 sessions in number.

Units: those whose peak channel lies in the area (VISp for V1; VISl, VISrl, VISal, VISpm, VISam for LM, RL, AL, PM, AM) and that pass the Allen quality criteria of the Methods (isi_violations < 0.5, amplitude_cutoff < 0.1, presence_ratio > 0.9). Counts are spikes in the 250 ms from each natural scenes onset (Methods). Sessions with more than 10% of trials silent in all V1 units are excluded (Methods).

| Claim | Tested here |
|---|---|
| data-v1-units, fig1b-across-neuron-slope-near-1, fig1b-across-stimulus-slope, figs3-higher-visual-areas | yes |
| figs4ab-noise-correlation-under-manipulation, fig1d-dimensionality-falls-with-slope, figs5c-space-occupancy-peak | yes |
| fig2a-overlap-rises-with-slope, fig2c-continuity-from-slope-1 | yes |
| fig3bc-geodesic-orthogonal-correlation, fig3d-correlation-peaks-near-1, fig3e-local-alignment-peaks-near-1, fig3f-global-alignment-peaks-near-1 | yes |
| fig4a-overlap-consistency-within-mice, fig4c-rsm-similarity-within-mice, fig4d-overlap-consistency-across-mice, fig4e-rsm-similarity-across-mice, figs6-higher-areas-consistency | yes |
| figs2-behavioral-state | yes (pupil in the sessions with eye tracking) |
| figs5a-movie-slope-near-1 | yes, on the Functional Connectivity sessions of dandiset 000022 |
| fig2b-decoding-falls-with-slope, figs4c-shuffling-alters-dependence | blocked as too large; the timing that decides it is at the end |
| figs1-gain-variance, fig2d-tangent-angle-rises-below-1 | blocked (attempt record) |

Seeds: each random result uses the seed of its claim, `zlib.crc32(b"kim-2025/<claim id>")`, and the RSM-based results are repeated under the seed plus 1000, 2000, 3000 and 4000.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
import time, zlib, itertools, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
from scipy.stats import mannwhitneyu, kruskal, spearmanr

import kim2025 as K

def wilcoxon(x, y=None, **kw):
    """scipy's Wilcoxon signed-rank test, with p = 1 when every difference is zero."""
    from scipy.stats import wilcoxon as _w
    d = np.asarray(x) - (0 if y is None else np.asarray(y))
    if np.all(d == 0):
        return type("R", (), {"pvalue": 1.0, "statistic": 0.0})()
    return _w(x, y, **kw) if y is not None else _w(x, **kw)

T0 = time.time()
pd.set_option("display.width", 220)
SLOPES = [float(s) for s in K.TARGET_SLOPES]
def seed(claim):
    return zlib.crc32(f"kim-2025/{claim}".encode())
paths = [p for p in K.asset_paths(K.ABO_ID, K.ABO_VERSION) if "probe" not in p]
print(f"dandiset {K.ABO_ID} version {K.ABO_VERSION}: {len(paths)} session files")

## Data check

In [ ]:
S = []
for p in paths:
    S.append(K.abo_session(p))
print(f"loaded in {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
rows = []
for i, s in enumerate(S):
    v1 = s["area"] == "V1"
    lab = s["labels"]
    _, n = np.unique(lab, return_counts=True)
    rows.append(dict(index=i, session=s["info"]["path"].split("/")[0], trials=s["info"]["trials"], stimuli=len(n),
                     repeats_min=n.min(), repeats_max=n.max(),
                     **{f"{a}_q": int((s["area"] == a).sum()) for a in K.ABO_AREAS},
                     V1_all=s["info"]["area_units_all"]["V1"],
                     silent_V1=round(K.silent_fraction(s["X"][:, v1]), 4),
                     speed_mean=round(float(np.nanmean(s["speed"])), 2), eye=s["info"]["eye_tracking"]))
tab = pd.DataFrame(rows).set_index("index")
tab["included"] = tab.silent_V1 <= K.SILENT_TRIAL_LIMIT
print(tab.to_string())
print("frame codes:", np.unique(S[1]["labels"])[[0, 1, -1]], "(-1 is the gray image)")
print("running speed unit:", {s["info"]["running_unit"] for s in S}, "; trial-mean speed range",
      f"{min(np.nanmin(s['speed']) for s in S):.1f} to {max(np.nanmax(s['speed']) for s in S):.1f}")
allc = np.concatenate([s["X"].ravel() for s in S])
print(f"spike counts per 250 ms: min {allc.min():.0f}, max {allc.max():.0f}, mean {allc.mean():.3f}")
keep = [i for i in tab.index if tab.loc[i, "included"]]
print("excluded sessions:", [i for i in tab.index if i not in keep], "; included:", len(keep))
print("duplicate check: distinct V1 count vectors", len({tuple(S[i]["X"][:, S[i]["area"] == "V1"].sum(0)) for i in tab.index}), "of", len(S))

Clock check: the image identity is decoded from V1 counts in the 250 ms after onset and in a window shifted 2 s later (several presentations later in the random sequence), in one session.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score
i0 = keep[0]
f = K.open_nwb(S[i0]["info"]["path"], K.ABO_ID, K.ABO_VERSION)
u = K.units_table(f)
rows_v1 = np.where((u.location == "VISp").to_numpy() & u.passes_quality.to_numpy())[0]
g = f["intervals/natural_scenes_presentations"]
fr, on = g["frame"][:].astype(int), g["start_time"][:]
for shift in (0.0, 2.0):
    C = K.spike_counts(f, rows_v1, on + shift, 0.25)
    acc = cross_val_score(LinearDiscriminantAnalysis(), C, fr, cv=StratifiedKFold(5, shuffle=False)).mean()
    print(f"session {i0}, window shifted {shift:+.1f} s: 119-way accuracy {acc:.3f} (chance {1/119:.4f})")
f.close()

## data-v1-units

> The main analysis uses V1 units from 30 sessions of 30 mice, 64 +- 25 units per session (14 to 110), 1920 units in all, with 119 stimuli per session (3570 stimulus fits).

In [ ]:
nv1 = tab.loc[keep, "V1_q"]
print(f"sessions {len(keep)} (of {len(S)}); V1 units per session mean {nv1.mean():.1f}, SD {nv1.std(ddof=1):.1f} (ddof 1; ddof 0 {nv1.std(ddof=0):.1f}), "
      f"range {nv1.min()} to {nv1.max()}, total {nv1.sum()}; stimuli per session {set(tab.loc[keep, 'stimuli'])}; fits {119 * len(keep)}")
print("mice: one session per subject folder,", tab.loc[keep, "session"].nunique(), "distinct subjects")
ng = pd.Series({i: int(((S[i]["area"] == "V1") & S[i]["good"]).sum()) for i in keep})
print(f"variant, also requiring the units table's quality label 'good': mean {ng.mean():.1f}, SD {ng.std(ddof=1):.1f}, range {ng.min()} to {ng.max()}, total {ng.sum()}")

AllenSDK, through which the paper read these data, keeps only units whose `quality` label is "good" in its default units table (`filter_by_validity=True`; allensdk/brain_observatory/ecephys/probes.py, lines 128 to 129, at commit a56bd8ea7613a812334602b64746cdd30c325a96), on top of the three criteria the Methods name. The NWB files hold all units with the label, so both readings are shown.

**Result: reproduced, resting on the AllenSDK reading of the unit selection.** Two of 32 sessions have more than 10% of trials silent in all V1 units (indices 0 and 9 here, the first and seventh by session id, which is how AllenSDK orders them; the code drops indices 0 and 6), leaving 30 sessions of 30 mice, each with 119 stimuli (3570 fits). With the Methods' three criteria alone there are 71.1 +- 27.1 V1 units per session (14 to 113), 2133 in all: the mean and the total are outside their tolerances (57.6 to 70.4; 1728 to 2112). Also requiring the label "good", as AllenSDK does by default, gives 66.9 +- 25.5 (14 to 112), 2008 in all, every value within tolerance.

## fig1b-across-neuron-slope-near-1 and fig1b-across-stimulus-slope

In [ ]:
fits = {i: K.fit_slopes(S[i]["X"][:, S[i]["area"] == "V1"], S[i]["labels"]) for i in keep}
an = pd.concat([fits[i].assign(session=i) for i in keep])
print(f"across-neuron slopes: n {len(an)}, mean {an.slope.mean():.4f}, SEM {an.slope.sem():.4f}, SD {an.slope.std(ddof=1):.4f}, "
      f"median {an.slope.median():.4f}, range {an.slope.min():.3f} to {an.slope.max():.3f}")
print("per-session mean slope range:", round(an.groupby('session').slope.mean().min(), 3), "to", round(an.groupby('session').slope.mean().max(), 3))
print("neurons per fit (mean > 0): median", an.n.median(), ", min", an.n.min())
an_g = pd.concat([K.fit_slopes(S[i]["X"][:, (S[i]["area"] == "V1") & S[i]["good"]], S[i]["labels"]) for i in keep])
print(f"variant, units also labelled 'good': n {len(an_g)}, mean {an_g.slope.mean():.4f}, SEM {an_g.slope.sem():.4f}, SD {an_g.slope.std(ddof=1):.4f}")

**Result (fig1b-across-neuron-slope-near-1): reproduced, resting on the AllenSDK reading of the unit selection.** With the units AllenSDK serves (quality criteria and label "good") the 3570 across-neuron slopes have mean 1.0489 (SEM 0.0011) and SD 0.0665, within 1.048 +- 0.001 and 0.07 +- 0.007. With the Methods' three criteria alone the mean is 1.0552 (SEM 0.0011), outside the tolerance, and the SD 0.0655 is within it. The slopes lie between 0.605 and 1.381.

In [ ]:
ast = np.concatenate([K.across_stimulus_slopes(S[i]["X"][:, S[i]["area"] == "V1"], S[i]["labels"]) for i in keep])
dropped = int(np.isnan(ast).sum()); ast = ast[~np.isnan(ast)]
mw = mannwhitneyu(ast, an.slope, alternative="two-sided")
print(f"across-stimulus slopes: n {len(ast)} ({dropped} neurons without 2 stimuli of nonzero mean dropped), mean {ast.mean():.4f}, SD {ast.std(ddof=1):.4f}, median {np.median(ast):.4f}")
print(f"across-neuron: mean {an.slope.mean():.4f}, median {an.slope.median():.4f}; rank-sum (Mann-Whitney U, two-sided) U = {mw.statistic:.0f}, p = {mw.pvalue:.3g}")
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(an.slope, bins=np.linspace(0, 3, 61), alpha=0.6, density=True, label="across neurons")
ax.hist(ast, bins=np.linspace(0, 3, 61), alpha=0.6, density=True, label="across stimuli"); ax.legend(); ax.set_xlabel("slope")
print("points outside 0 to 3:", int((an.slope < 0).sum() + (an.slope > 3).sum()), "across-neuron,", int((ast < 0).sum() + (ast > 3).sum()), "across-stimulus")
plt.show()

**Result (fig1b-across-stimulus-slope): reproduced.** The 2133 across-stimulus slopes have mean 1.134 and SD 0.206 (expected 1.13 +- 0.21 and 0.21 +- 0.021), larger than the across-neuron slopes in mean (1.134 against 1.055) and median (1.149 against 1.058), and the two-sided rank-sum test gives p = 5e-105. These use the Methods' three unit criteria; the "good" reading was not run for this claim.

## figs2-behavioral-state

> The across-neuron slope is slightly but significantly modulated by behavioral state, smallest when the mouse runs and when its pupils are dilated, while staying near 1.

Locomotion: trial mean running speed >= 1 cm/s is moving; M1 per session, stimulus and state, pooled; stimulus and state cells with fewer than 2 trials cannot give a variance and are dropped. Pupil: trial mean of the blink-filtered pupil area (`processing/filtered_gaze_mapping/pupil_area`), present in the sessions marked `eye` in the data check; tertiles over the trials of all sessions pooled (code, fit_slope_histogram.ipynb cell 67), per-session tertiles as a variant; Kruskal-Wallis across the three groups of slopes.

In [ ]:
st_rows = []
for i in keep:
    X, lab, sp = S[i]["X"][:, S[i]["area"] == "V1"], S[i]["labels"], S[i]["speed"]
    for state, sel in (("moving", sp >= 1), ("stationary", sp < 1)):
        for c in np.unique(lab):
            m = (lab == c) & sel
            if m.sum() < 2:
                continue
            Y = X[m]; mu, v = Y.mean(0), Y.var(0, ddof=1); ok = (mu > 0) & (v > 0)
            if ok.sum() >= 2:
                st_rows.append((i, state, c, np.polyfit(np.log10(mu[ok]), np.log10(v[ok]), 1)[0], m.sum()))
bs = pd.DataFrame(st_rows, columns=["session", "state", "stim", "slope", "trials"])
print(bs.groupby("state").slope.describe().round(4).to_string())
mv, stn = bs[bs.state == "moving"].slope, bs[bs.state == "stationary"].slope
print(f"moving vs stationary: medians {mv.median():.4f} vs {stn.median():.4f}; Mann-Whitney two-sided p = {mannwhitneyu(mv, stn).pvalue:.3g}")
print("sessions with moving fits:", bs[bs.state == "moving"].session.nunique(), "; fraction of trials moving:",
      round(float(np.mean(np.concatenate([S[i]["speed"] >= 1 for i in keep]))), 3))
pup = {i: K.abo_pupil(S[i]["info"]["path"], K.abo_onsets(S[i]["info"]["path"])) for i in keep}
pup = {i: v for i, v in pup.items() if v is not None}
allp = np.concatenate([v for v in pup.values()])
edges = np.nanpercentile(allp, [100 / 3, 200 / 3])
print(f"pupil: {len(pup)} included sessions with eye tracking; trials without a pupil value {np.isnan(allp).mean():.3f}; "
      f"pooled tertile edges {edges[0]:.4f} and {edges[1]:.4f}; range {np.nanmin(allp):.4g} to {np.nanmax(allp):.4g} (labelled pixels^2)")
for label, per_session in (("pooled tertiles (code)", False), ("per-session tertiles (variant)", True)):
    pr = []
    for i, pv in pup.items():
        X, lab = S[i]["X"][:, S[i]["area"] == "V1"], S[i]["labels"]
        e = np.nanpercentile(pv, [100 / 3, 200 / 3]) if per_session else edges
        tert = np.where(np.isnan(pv), -1, np.digitize(pv, e))
        for t_ in range(3):
            for c in np.unique(lab):
                m = (lab == c) & (tert == t_)
                if m.sum() < 2:
                    continue
                Y = X[m]; mu, v = Y.mean(0), Y.var(0, ddof=1); ok = (mu > 0) & (v > 0)
                if ok.sum() >= 2:
                    pr.append((i, t_, np.polyfit(np.log10(mu[ok]), np.log10(v[ok]), 1)[0]))
    pdf = pd.DataFrame(pr, columns=["session", "tertile", "slope"])
    meds = pdf.groupby("tertile").slope.median()
    kw = kruskal(*[pdf[pdf.tertile == t_].slope for t_ in range(3)])
    print(f"{label}: fits per tertile {pdf.groupby('tertile').size().to_dict()}, median slope per tertile {meds.round(4).to_dict()}, "
          f"largest tertile lowest: {bool(meds.idxmin() == 2)}, Kruskal-Wallis p = {kw.pvalue:.3g}, medians in 0.9-1.1: {bool(meds.between(0.9, 1.1).all())}")
print("per-session median slope, moving minus stationary:",
      (bs.groupby(["session", "state"]).slope.median().unstack().dropna().pipe(lambda d: d.moving - d.stationary)).describe().round(4).to_dict())

**Result: reproduced.** Moving trials (44.6% of trials) give lower slopes than stationary ones (medians 0.999 against 1.067, rank-sum p = 1.5e-303). Over the 25 included sessions with eye tracking, the largest pupil tertile has the lowest median slope (1.039, against 1.077 and 1.053), Kruskal-Wallis p = 2e-46 with pooled tertiles (per-session tertiles, 1.043 against 1.075 and 1.053, p = 3e-42). Every state's median slope lies between 0.9 and 1.1. Five included sessions have no eye tracking, and 10.5% of trials in the others have no pupil value.

## figs3-higher-visual-areas

> The across-neuron slope is concentrated around 1 in the five higher visual areas as in V1.

M1 per area in every session with at least two quality units in the area (a line needs two neurons), over all 32 sessions and over the 30 included sessions.

In [ ]:
for label, idx in (("all 32 sessions", list(tab.index)), ("30 included sessions", keep)):
    out = {}
    for a in K.ABO_AREAS:
        sl, ns = [], 0
        for i in idx:
            m = S[i]["area"] == a
            if m.sum() >= 2:
                ns += 1
                sl.append(K.fit_slopes(S[i]["X"][:, m], S[i]["labels"]).slope.to_numpy())
        sl = np.concatenate(sl)
        out[a] = dict(sessions=ns, sessions_any_unit=int(sum((S[i]["area"] == a).sum() > 0 for i in idx)), fits=len(sl),
                      median=round(float(np.median(sl)), 4), mean=round(float(sl.mean()), 4),
                      within_0_9_1_1=bool(0.9 <= np.median(sl) <= 1.1))
    print(label); print(pd.DataFrame(out).T.to_string())

**Result: reproduced.** Over all 32 sessions, the sessions with at least two quality units are LM 24, RL 28, AL 24, PM 20 and AM 26, each within tolerance of 24, 27, 24, 20 and 26, and the median slopes are 1.058, 1.061, 1.046, 1.004 and 1.026, all between 0.9 and 1.1 (V1 1.063). Over the 30 included sessions the counts are 22, 26, 23, 18 and 25 and the medians again lie between 0.9 and 1.1.

## Measures under slope manipulation

For every included session, V1 units, the as-is data and the 21 target slopes 0 to 2, `K.session_measures` computes the effective dimensionality, the PC1 fraction, pairwise noise correlations, the overlap matrices (Methods and code thresholds), occupied voxels, geodesic distances and orthogonal variances, local and global alignment, the RSM, and the half-split consistencies; and, for LM and PM, the RSM and its half-split consistency. Sessions run in 4 processes.

In [ ]:
SEEDS = {k: seed(k) for k in ["fig4e-rsm-similarity-across-mice", "fig4c-rsm-similarity-within-mice",
                              "fig4a-overlap-consistency-within-mice", "figs6-higher-areas-consistency"]}
def job(i, seeds=SEEDS, halves=True):
    s = S[i]
    areas = {a: s["X"][:, s["area"] == a] for a in ("LM", "PM") if (s["area"] == a).sum() >= 2}
    return K.session_measures(s["X"][:, s["area"] == "V1"], s["labels"], seeds, areas=areas, do_halves=halves)
t = time.time()
with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
    M = dict(zip(keep, ex.map(job, keep)))
print(f"measures for {len(M)} sessions in {(time.time() - t) / 60:.1f} min")
CONDS = ["as-is"] + SLOPES

def curve(key, f=lambda v: v):
    return pd.DataFrame({i: {c: f(M[i][key][c]) for c in CONDS} for i in M}).T

def o1(df, label, n_label="sessions"):
    m = df[SLOPES].mean()
    peak = m.idxmax()
    p0, p2 = wilcoxon(df[0.0], df["as-is"]).pvalue, wilcoxon(df[2.0], df["as-is"]).pvalue
    lower = bool(m[0.0] < df["as-is"].mean() and m[2.0] < df["as-is"].mean())
    res = dict(peak_slope=peak, peak_mean=round(m.max(), 4), mean_asis=round(df["as-is"].mean(), 4), mean_slope0=round(m[0.0], 4),
               mean_slope1=round(m[1.0], 4), mean_slope2=round(m[2.0], 4), p_slope0=float(f"{p0:.3g}"), p_slope2=float(f"{p2:.3g}"),
               ends_lower=lower, reading_a=bool(0.8 <= peak <= 1.2 and lower and p0 < 0.05 and p2 < 0.05),
               reading_b=bool(0.7 <= peak <= 1.3 and lower and p0 < 0.05 and p2 < 0.05))
    print(f"{label} (n = {len(df)} {n_label}):", res)
    print("   curve:", " ".join(f"{c}:{v:.3f}" for c, v in m.items()))
    return res

def o2(df, direction, label):
    m = df[SLOPES].mean()
    a0, a1, a2, asis = m[0.0], m[1.0], m[2.0], df["as-is"].mean()
    order = (a2 < a1 < a0) if direction == "decrease" else (a2 > a1 > a0)
    p0, p2 = wilcoxon(df[0.0], df["as-is"]).pvalue, wilcoxon(df[2.0], df["as-is"]).pvalue
    steps = np.diff(m.to_numpy())
    strict = bool(np.all(steps < 0)) if direction == "decrease" else bool(np.all(steps > 0))
    res = dict(mean_asis=round(asis, 4), mean_slope0=round(a0, 4), mean_slope1=round(a1, 4), mean_slope2=round(a2, 4),
               order=order, p_slope0=float(f"{p0:.3g}"), p_slope2=float(f"{p2:.3g}"), strict_every_step=strict,
               holds=bool(order and p0 < 0.05 and p2 < 0.05))
    print(f"{label} (n = {len(df)} sessions):", res)
    print("   curve:", " ".join(f"{c}:{v:.3f}" for c, v in m.items()))
    return res

## figs4ab-noise-correlation-under-manipulation

> Rescaling residuals to change the slope leaves pairwise noise correlations unchanged but changes population-wise noise correlation.

In [ ]:
ncd = pd.DataFrame({i: M[i]["nc_maxdiff"] for i in M}).T
print("largest |noise correlation change| over sessions, stimuli and pairs, at slopes 0, 1, 2:", ncd.max().to_dict())
pc = curve("pc1")
for sl in (0.0, 2.0):
    frac = [np.mean(np.abs(M[i]["pc1"][sl] - M[i]["pc1"]["as-is"]) > 1e-9) for i in M]
    print(f"slope {sl}: fraction of stimuli whose PC1 variance fraction differs from as-is, per session min {min(frac):.3f}, mean {np.mean(frac):.3f}; "
          f"pooled mean PC1 fraction {np.mean([np.nanmean(M[i]['pc1'][sl]) for i in M]):.4f} vs as-is {np.mean([np.nanmean(M[i]['pc1']['as-is']) for i in M]):.4f}")
ex7 = keep[7] if len(keep) > 7 else keep[0]
print(f"example (session index {ex7}): mean PC1 fraction as-is {np.nanmean(M[ex7]['pc1']['as-is']):.4f}, slope 0 {np.nanmean(M[ex7]['pc1'][0.0]):.4f}, slope 2 {np.nanmean(M[ex7]['pc1'][2.0]):.4f}")

**Result: reproduced.** Over the 30 sessions, every stimulus's pairwise noise correlations at slopes 0, 1 and 2 equal the as-is values to within 7.3e-15, and the fraction of variance on the first principal component differs from as-is for every stimulus at slopes 0 and 2 (pooled mean 0.160 at slope 0 and 0.505 at slope 2, against 0.333 as-is).

## fig1d-dimensionality-falls-with-slope

In [ ]:
dim = curve("dim", np.mean)
r_dim = o2(dim, "decrease", "effective dimensionality")

**Result: reproduced.** The mean effective dimensionality falls at every 0.1 step from 0.282 at slope 0 to 0.124 at slope 1 and 0.061 at slope 2 (as-is 0.120); slopes 0 and 2 differ from as-is by the Wilcoxon signed-rank test (p = 1.9e-09 each).

## figs5c-space-occupancy-peak

In [ ]:
vox = curve("vox").astype(float)
r_vox = o1(vox, "occupied unit voxels")
print("sessions where the as-is count equals the number of trials (every trial in its own voxel):",
      int(sum(vox.loc[i, 'as-is'] == len(S[i]['labels']) for i in M)), "of", len(M))
print("sessions whose maximum over slopes is reached at more than one slope:", int(sum((vox.loc[i, SLOPES] == vox.loc[i, SLOPES].max()).sum() > 1 for i in M)))

**Result: reproduced.** The mean count of occupied unit voxels peaks at slope 0.8 (5949.9), within both windows, and is lower at slope 0 (5674.3) and slope 2 (340.7) than as-is (5949.9), with p = 1.7e-06 for each. The measure is at its ceiling near the peak: in 28 of 30 sessions every as-is trial occupies a voxel of its own, and the maximum is shared by more than one slope, so the peak at 0.8 marks the middle of a plateau that runs from about 0.7 to 1.0.

## fig2a-overlap-rises-with-slope and fig2c-continuity-from-slope-1

In [ ]:
ov = curve("O_m", K.mean_offdiag)
r_ov = o2(ov, "increase", "overlap, distinct pairs, Methods threshold (main)")
ov_code = curve("O_c", lambda O: float(O.mean()))
r_ov_code = o2(ov_code, "increase", "overlap, code variant (code threshold, full matrix with diagonal)")
ov_c2 = curve("O_c", K.mean_offdiag)
o2(ov_c2, "increase", "overlap, code threshold, distinct pairs")

**Result (fig2a-overlap-rises-with-slope): reproduced.** The mean overlap over ordered pairs of distinct stimuli rises at every 0.1 step from 0.062 at slope 0 to 0.276 at slope 1 and 0.425 at slope 2 (as-is 0.286), with p = 1.9e-09 for slopes 0 and 2 against as-is. The code's variant (its threshold, the full matrix with the diagonal) gives 0.047, 0.222 and 0.321 (p = 1.9e-09 and 3.2e-07), also rising at every step.

In [ ]:
scc = curve("O_m", K.largest_scc).astype(float)
scc_c = curve("O_c", K.largest_scc).astype(float)
for name, df in (("Methods threshold", scc), ("code threshold", scc_c)):
    m = df[SLOPES].mean()
    full = [c for c in SLOPES if m[c] >= 119]
    first = full[0] if full else None
    try:
        p0 = wilcoxon(df[0.0], df["as-is"]).pvalue
    except ValueError:
        p0 = float("nan")
    print(f"{name}: mean largest SCC size by slope", " ".join(f"{c}:{v:.1f}" for c, v in m.items()), f"| as-is {df['as-is'].mean():.1f}")
    print(f"   first slope with mean 119: {first}; reading a (0.8-1.2): {first is not None and 0.8 <= first <= 1.2}; reading b (0.7-1.3): {first is not None and 0.7 <= first <= 1.3}; "
          f"slope 0 mean {m[0.0]:.1f}, Wilcoxon vs as-is p = {p0:.3g}; sessions at 119 as-is {int((df['as-is'] == 119).sum())} of {len(df)}")
    print("   sessions reaching 119 by slope:", " ".join(f"{c}:{int((df[c] == 119).sum())}" for c in SLOPES))

**Result (fig2c-continuity-from-slope-1): reproduced.** The mean size of the largest strongly connected component first reaches 119 at slope 1.1 (Methods threshold; 1.2 with the code's threshold), inside both windows; at slope 0 it is 107.2 (103.0) and differs from as-is (118.9, 29 of 30 sessions complete) with p = 5.7e-05 (4.0e-05). The number of sessions whose component covers all stimuli grows from 9 at slope 0 to 29 at slope 1 and 30 from 1.1.

## fig3bc-geodesic-orthogonal-correlation, fig3d, fig3e, fig3f

In [ ]:
geo = curve("geo_rho")
rho = geo["as-is"]
w = wilcoxon(rho, alternative="greater")
print(f"as-is Spearman rho per session: mean {rho.mean():.4f}, range {rho.min():.4f} to {rho.max():.4f}, positive in {(rho > 0).sum()} of {len(rho)}; "
      f"one-sided Wilcoxon against 0 p = {w.pvalue:.3g}; pairs per session {int(np.isfinite(K._offdiag(M[keep[0]]['asis_ortho'])).sum())}")
print("sessions with non-finite geodesics:", int(sum((~np.isfinite(M[i]["asis_geo"])).any() for i in M)))

**Result (fig3bc-geodesic-orthogonal-correlation): reproduced.** The as-is Spearman correlation between inter-centroid geodesic distance and orthogonal variance over the 14042 ordered pairs is positive in all 30 sessions (mean 0.323, 0.034 to 0.558), and the one-sided Wilcoxon test against 0 gives p = 9.3e-10.

In [ ]:
r_geo = o1(geo, "geodesic-orthogonal variance Spearman")

**Result (fig3d-correlation-peaks-near-1): reproduced.** The mean correlation peaks at slope 1.2 (0.330), inside both windows; it is lower at slope 0 (0.223) and slope 2 (0.305) than as-is (0.323), with p = 2.6e-07 and 0.011.

In [ ]:
r_loc = o1(curve("local"), "local alignment (as-is neighbour, code)")
o1(curve("local_slope_nn"), "local alignment (variant, neighbour from the slope-changed data)")

**Result (fig3e-local-alignment-peaks-near-1): reproduced.** Mean local alignment peaks at slope 1.0 (0.830); slope 0 (0.728) and slope 2 (0.804) are lower than as-is (0.830), p = 1.0e-06 and 0.0047. With the neighbour taken from the slope-changed data (variant) the peak is also at 1.0.

In [ ]:
r_glo = o1(curve("global"), "global alignment")

**Result (fig3f-global-alignment-peaks-near-1): reproduced.** Mean global alignment peaks at slope 1.1 (0.844); slope 0 (0.697) and slope 2 (0.806) are lower than as-is (0.844), p = 8.0e-08 and 0.00046.

## fig4a, fig4c, fig4d, fig4e, figs6

In [ ]:
r_4a = o1(curve("ov_half"), "overlap consistency within session, full matrices (code, main)")
o1(curve("ov_half_off"), "overlap consistency within session, off-diagonal (variant)")

**Result (fig4a-overlap-consistency-within-mice): reproduced.** The mean Spearman correlation between the overlap matrices of two random halves of the neurons peaks at slope 0.9 (0.650, with 0.650 at 1.0); slope 0 (0.419) and slope 2 (0.369) are lower than as-is (0.643), p = 1.9e-09 each. The off-diagonal variant also peaks at 0.9. Only the claim's seed was run for this measure.

In [ ]:
r_4c = o1(curve("rsm_half"), "RSM similarity within session")

**Result (fig4c-rsm-similarity-within-mice): reproduced.** The mean half-split RSM correlation peaks at slope 1.1 (0.448); slope 0 (0.418) and slope 2 (0.231) are lower than as-is (0.471), p = 1.2e-04 and 3.5e-08. The as-is value is higher than at any changed slope. Under the four further seeds the peak is at 1.1 each time (seed check below).

In [ ]:
pairs = list(itertools.combinations(keep, 2))
def across(key, f):
    return pd.DataFrame({p: {c: f(M[p[0]][key][c], M[p[1]][key][c]) for c in CONDS} for p in pairs}).T
full_sp = lambda A, B: spearmanr(A.ravel(), B.ravel()).statistic
off_sp = lambda A, B: spearmanr(K._offdiag(A), K._offdiag(B)).statistic
r_4d = o1(across("O_m", full_sp), "overlap consistency across sessions, full matrices (main)", "session pairs")
o1(across("O_m", off_sp), "overlap consistency across sessions, off-diagonal (variant)", "session pairs")

**Result (fig4d-overlap-consistency-across-mice): reproduced.** Over the 435 session pairs, the mean Spearman correlation between overlap matrices peaks at slope 1.2 (0.431); slope 0 (0.248) and slope 2 (0.352) are lower than as-is (0.412), p = 5.7e-70 and 6.3e-31. The off-diagonal variant also peaks at 1.2.

In [ ]:
rsm_sp = lambda A, B: spearmanr(A.ravel(), B.ravel(), nan_policy="omit").statistic
r_4e = o1(across("rsm", rsm_sp), "RSM similarity across sessions", "session pairs")

**Result (fig4e-rsm-similarity-across-mice): reproduced.** Over the 435 session pairs the mean RSM correlation peaks at slope 1.1 (0.397); slope 0 (0.356) and slope 2 (0.185) are lower than as-is (0.406), p = 1.9e-51 and 3.3e-72. The as-is value is higher than at any changed slope. Under the four further seeds the peak is at 1.1 each time.

In [ ]:
def near_max(df, n_label):
    m, se = df[SLOPES].mean(), df[SLOPES].sem()
    peak = m.idxmax()
    a = bool(m[1.0] >= m.max() - se[peak])
    b = bool(0.8 <= peak <= 1.2)
    b2 = bool(0.7 <= peak <= 1.3)
    return dict(n=len(df), peak=peak, mean_at_1=round(m[1.0], 4), max=round(m.max(), 4), sem_at_peak=round(se[peak], 4),
                reading_a=a, reading_b_08_12=b, reading_b_07_13=b2)
s6 = {}
for a in ("LM", "PM"):
    ids = [i for i in M if a in M[i]["area_rsm_half"]]
    within = pd.DataFrame({i: M[i]["area_rsm_half"][a] for i in ids}).T
    pa = list(itertools.combinations(ids, 2))
    acr = pd.DataFrame({p: {c: rsm_sp(M[p[0]]["area_rsm"][a][c], M[p[1]]["area_rsm"][a][c]) for c in CONDS} for p in pa}).T
    s6[(a, "S6B within")] = near_max(within, "sessions")
    s6[(a, "S6C across")] = near_max(acr, "pairs")
    print(a, "within curve:", " ".join(f"{c}:{v:.3f}" for c, v in within[SLOPES].mean().items()))
    print(a, "across curve:", " ".join(f"{c}:{v:.3f}" for c, v in acr[SLOPES].mean().items()))
print(pd.DataFrame(s6).T.to_string())

**Result (figs6-higher-areas-consistency): reproduced under reading (a).** For LM and PM, within sessions (S6B; 22 and 18 sessions) and across session pairs (S6C; 231 and 153 pairs), the mean at slope 1 lies within one SEM of the curve's maximum in all four panels (reading a). Under reading (b) the maximum lies in the window only across pairs (LM 1.2, PM 1.0); within sessions the curves are nearly flat from slope 0 to 1.1 and peak at 0.3 (LM) and 0.4 (PM). Reading (a) holds in all four panels under every seed checked below.

### Seed check for the RSM-based results (fig4c, fig4e, figs6)

In [ ]:
def rjob(args):
    i, off = args
    s = S[i]
    areas = {a: s["X"][:, s["area"] == a] for a in ("LM", "PM") if (s["area"] == a).sum() >= 2}
    return K.random_measures(s["X"][:, s["area"] == "V1"], s["labels"], {k: v + off for k, v in SEEDS.items()}, areas=areas)
t = time.time()
seed_res = {}
for off in (1000, 2000, 3000, 4000):
    with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
        seed_res[off] = dict(zip(keep, ex.map(rjob, [(i, off) for i in keep])))
print(f"seed checks in {(time.time() - t) / 60:.1f} min")

In [ ]:
rows = []
for off, R_ in seed_res.items():
    c4 = pd.DataFrame({i: R_[i]["rsm_half"] for i in R_}).T[SLOPES].mean()
    e4 = pd.DataFrame({p: {c: rsm_sp(R_[p[0]]["rsm"][c], R_[p[1]]["rsm"][c]) for c in SLOPES} for p in pairs}).T.mean()
    row = dict(offset=off, fig4c_peak=c4.idxmax(), fig4c_max=round(c4.max(), 4), fig4e_peak=e4.idxmax(), fig4e_max=round(e4.max(), 4))
    for a in ("LM", "PM"):
        ids = [i for i in R_ if a in R_[i]["area_rsm_half"]]
        wa = pd.DataFrame({i: R_[i]["area_rsm_half"][a] for i in ids}).T[SLOPES]
        pa = list(itertools.combinations(ids, 2))
        aa = pd.DataFrame({p: {c: rsm_sp(R_[p[0]]["area_rsm"][a][c], R_[p[1]]["area_rsm"][a][c]) for c in SLOPES} for p in pa}).T
        row[f"{a}_within_peak"], row[f"{a}_within_1_vs_max_minus_sem"] = wa.mean().idxmax(), bool(wa.mean()[1.0] >= wa.mean().max() - wa.sem()[wa.mean().idxmax()])
        row[f"{a}_across_peak"], row[f"{a}_across_1_vs_max_minus_sem"] = aa.mean().idxmax(), bool(aa.mean()[1.0] >= aa.mean().max() - aa.sem()[aa.mean().idxmax()])
    rows.append(row)
print(pd.DataFrame(rows).set_index("offset").to_string())
print(f"notebook time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## figs5a-movie-slope-near-1

> In the natural movie data the across-neuron log-mean against log-variance relation is also linear with slope near 1.

The paper's natural movie data are the Allen "Functional Connectivity" sessions, deposited as dandiset 000022, version 0.251116.2247 (the only published version; doi:10.48324/dandi.000022/0.251116.2247), streamed. In these files the 60-repeat block of natural movie one is `natural_movie_one_more_repeats_presentations` (a 20-repeat shuffled version is a separate table). Each repeat is cut into 75 bins of 12 frames; V1 units by the Allen quality criteria; M1 per bin and session. Sessions are excluded when a repeat is incomplete (the Methods' "truncated movies") or when they have no V1 units.

In [ ]:
fc_paths = [p for p in K.asset_paths(K.FC_ID, K.FC_VERSION) if "probe" not in p]
mv_rows, mv_slopes = [], []
for p in fc_paths:
    Xm, labm, repm = K.movie_session(p)
    ok = repm["repeats_incomplete"] == 0 and repm["v1_units"] >= 2
    mv_rows.append(dict(session=p.split("/")[0], **{k: v for k, v in repm.items() if k != "path"}, used=ok))
    if ok:
        mv_slopes.append(K.fit_slopes(Xm, labm).slope.to_numpy())
mvt = pd.DataFrame(mv_rows)
print(mvt.to_string())
ms = np.concatenate(mv_slopes)
print(f"sessions {len(fc_paths)}, used {int(mvt.used.sum())}; bins per session {set(len(x) for x in mv_slopes)}; fits {len(ms)}")
print(f"pooled slopes: median {np.median(ms):.4f}, mean {ms.mean():.4f}, SD {ms.std(ddof=1):.4f}, range {ms.min():.3f} to {ms.max():.3f}")
print(f"reading 1 (median in 0.9 to 1.1): {0.9 <= np.median(ms) <= 1.1}; reading 2 (0.85 to 1.15): {0.85 <= np.median(ms) <= 1.15}")

**Result: partially reproduced.** Of the 26 Functional Connectivity sessions, none has an incomplete repeat of the movie and 2 have no V1 units, so 24 are used where the paper used 23 (the expected count is exact); each gives 75 bins. The 1800 pooled slopes have median 1.131: outside 0.9 to 1.1 (reading 1) and inside 0.85 to 1.15 (reading 2), so the slope is near 1 under one of the criterion's readings, while the session count differs by one.

## Timing that blocks fig2b-decoding-falls-with-slope (and figs4c-shuffling-alters-dependence)

The decoding of Figure 2B (sklearn `SVC(kernel="linear")`, 119 classes, 10 folds, RSA_decoding.py `decode_ABO`) is timed here on one included session, as-is and at slope 2, to estimate the whole: 30 sessions times 22 conditions.

In [ ]:
i1 = keep[0]
times = {}
for c in (None, 2.0):
    t = time.time()
    a = K.decode_session(S[i1]["X"][:, S[i1]["area"] == "V1"], S[i1]["labels"], c, seed("fig2b-decoding-falls-with-slope"))
    times[c] = time.time() - t
    print(f"session {i1}, {'as-is' if c is None else f'slope {c}'}: accuracy {a:.3f}, {times[c]:.0f} s")
lo, hi = 22 * len(keep) * times[None], 22 * len(keep) * times[2.0]
print(f"estimate for {22 * len(keep)} session-conditions: {lo / 3600:.1f} to {hi / 3600:.1f} CPU hours, "
      f"{lo / 4 / 60:.0f} to {hi / 4 / 60:.0f} min on 4 CPUs; Figure S4C doubles it with the shuffled data")
print(f"notebook time {(time.time() - T0) / 60:.1f} min")

**Result (fig2b-decoding-falls-with-slope): blocked, too large.** One decoding (10 folds) takes about 11 s as-is and about 110 s at slope 2 in one session (the exact times printed above vary from run to run), so the 660 session-conditions of Figure 2B would need about 2 to 20 CPU hours, about 30 to 300 minutes on the runner's 4 CPUs, over the sixty-minute limit of a notebook; Figure S4C (figs4c-shuffling-alters-dependence) repeats it on shuffled data.

## Summary

All claims here are `data: elsewhere`: they are tested on the Allen data as deposited in dandisets 000021 and 000022, not on dandiset 000248.

| Claim | Outcome |
|---|---|
| data-v1-units | reproduced (assumed: AllenSDK's default "good" unit label) |
| fig1b-across-neuron-slope-near-1 | reproduced (same assumption) |
| fig1b-across-stimulus-slope | reproduced |
| figs2-behavioral-state | reproduced |
| figs3-higher-visual-areas | reproduced |
| figs4ab-noise-correlation-under-manipulation | reproduced |
| fig1d-dimensionality-falls-with-slope | reproduced |
| figs5c-space-occupancy-peak | reproduced |
| fig2a-overlap-rises-with-slope | reproduced |
| fig2c-continuity-from-slope-1 | reproduced |
| fig3bc-geodesic-orthogonal-correlation | reproduced |
| fig3d-correlation-peaks-near-1 | reproduced |
| fig3e-local-alignment-peaks-near-1 | reproduced |
| fig3f-global-alignment-peaks-near-1 | reproduced |
| fig4a-overlap-consistency-within-mice | reproduced |
| fig4c-rsm-similarity-within-mice | reproduced |
| fig4d-overlap-consistency-across-mice | reproduced |
| fig4e-rsm-similarity-across-mice | reproduced |
| figs6-higher-areas-consistency | reproduced (reading a) |
| figs5a-movie-slope-near-1 | partially reproduced |
| fig2b-decoding-falls-with-slope | blocked, too large |
| figs4c-shuffling-alters-dependence | blocked, too large (depends on fig2b) |
| figs1-gain-variance, fig2d-tangent-angle-rises-below-1 | blocked (not run in this attempt; see the record) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the data are read from the DANDI copies of the Allen release with h5py, not through AllenSDK. The units follow the Methods' three quality criteria in every analysis; AllenSDK's default also drops units not labelled "good" (125 of 2133 V1 units), which is shown as a reading for the two counting claims only. The slope fits, slope changes, overlap, geodesic distances, orthogonal variance, alignment and RSMs follow the claims file and the paper's code at commit ac3300b, including the code's as-is nearest neighbour for local alignment. The random neuron partitions and trial orders use this attempt's seeds, not the code's `np.random.seed(0)` sequence.

What was found about the data: the 32 Brain Observatory 1.1 sessions of dandiset 000021 match the paper's 32, the exclusion rule removes the same two sessions as the paper's code, the sessions per higher visual area match the paper, and the V1 unit counts and the across-neuron slope match it once AllenSDK's "good" label is applied. In dandiset 000022, 24 of 26 sessions have V1 units and complete movie repeats, one more than the paper's 23. The pupil area in `filtered_gaze_mapping` is labelled "Pixels ^ 2" but holds values of the order of 0.001 (pooled tertile edges 0.0016 and 0.0026), too small for an area in pixels. Five of the 30 included sessions have no eye tracking.